# spec_rebuild_foundation_v1

This research card verifies the first reusable `src/` rebuild slice against the local qlib calendar and canonical SOXL panel. It writes card-local sidecars for dashboard, notebook, and promotion readers.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

root = Path.cwd().resolve()
while not (root / 'AGENTS.md').exists():
    if root.parent == root:
        raise RuntimeError('repo root not found')
    root = root.parent

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.analysis import AnalysisReport, assign_regimes, compare_model_metrics, compute_window_metrics
from src.backtest import run_costed_backtest
from src.data import canonical_snapshot_from_panel, load_qlib_calendar

card_dir = root / 'research' / 'spec_rebuild_foundation_v1'
calendar = load_qlib_calendar(root / 'data' / 'qlib' / 'soxl')
panel_path = root / 'data' / 'processed' / 'panels' / 'soxl_soxx_vix_full.csv'
panel = pd.read_csv(panel_path, parse_dates=['date'])
panel = panel[['date', 'soxl_close', 'soxl_volume', 'vix']].rename(
    columns={'soxl_close': 'close', 'soxl_volume': 'volume'}
)

snapshot = canonical_snapshot_from_panel(
    panel,
    calendar=calendar,
    instrument='SOXL',
    as_of='2026-06-09',
    source_path=panel_path,
)
daily = snapshot.data.copy()
daily['signal'] = (daily['close'].pct_change(5).fillna(0.0) > 0).astype(float)
backtest = run_costed_backtest(daily, price_col='close', signal_col='signal', cost_bps=10, slippage_bps=5)
regime_frame = assign_regimes(backtest.assign(vix=daily['vix'], volume=daily['volume']), price_col='close')
window_metrics = compute_window_metrics(
    backtest,
    windows={
        '2026_ytd': ('2026-01-02', '2026-06-09'),
        'june_2026': ('2026-06-01', '2026-06-09'),
    },
)
model_comparison = compare_model_metrics(
    pd.DataFrame(
        [
            {'model': 'mean_return_smoke', 'return_pct': 12.0, 'max_drawdown_pct': -3.0, 'turnover': 0.2},
            {'model': 'buy_hold_reference', 'return_pct': 20.0, 'max_drawdown_pct': -15.0, 'turnover': 0.0},
        ]
    )
)
report = AnalysisReport(
    name='spec_rebuild_foundation_v1',
    summary={
        'source': snapshot.attrs['data_source'],
        'rows': int(len(daily)),
        'as_of': snapshot.as_of,
        'top_model': str(model_comparison.iloc[0]['model']),
    },
    regime_metrics=regime_frame[['date', 'vix_regime', 'trend_regime', 'volume_regime', 'drawdown_regime', 'volatility_regime', 'market_regime']],
    window_metrics=window_metrics,
    model_comparison=model_comparison,
    failure_modes=['foundation-only-no-promotion', 'smoke-model-not-production-signal'],
)
paths = report.write_sidecars(card_dir)

conclusion = '''# 中文结论

- 做了什么：使用 qlib calendar 和 canonical SOXL panel 跑通 `src/` 基础分析切片。
- 核心指标：生成 window、regime、model comparison 三组 sidecar，top model 为 mean_return_smoke。
- regime 差异：产物按 VIX、趋势、回撤、波动率、成交量冲击和牛熊/震荡切片，供后续模型稳定性分析复用。
- 是否晋升：不晋升；这是框架烟测卡，不是生产策略。
- 风险：样本和模型都只是基础验证，不能作为交易信号。
- 下一步：接入真实 QuantBench model registry、objective promotion score 和 production paper/sim notebook。
'''
(card_dir / 'conclusion.md').write_text(conclusion, encoding='utf-8')

summary = {
    'experiment': 'spec_rebuild_foundation_v1',
    'source': snapshot.attrs['data_source'],
    'qlib_calendar_size': snapshot.attrs['qlib_calendar_size'],
    'as_of': snapshot.as_of,
    'artifacts': sorted(path.name for path in card_dir.iterdir() if path.name not in {'spec_rebuild_foundation_v1.ipynb', 'summary.json'}),
}
(card_dir / 'summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True), encoding='utf-8')
summary


{'experiment': 'spec_rebuild_foundation_v1',
 'source': 'qlib_calendar_plus_canonical_snapshot',
 'qlib_calendar_size': 4088,
 'as_of': '2026-06-09',
 'artifacts': ['analysis_summary.json',
  'conclusion.md',
  'failure_modes.md',
  'model_comparison.csv',
  'regime_metrics.csv',
  'window_metrics.csv']}

## 中文结论

- 做了什么：使用 qlib calendar 和 canonical SOXL panel 跑通 `src/` 基础分析切片。
- 核心指标：生成 window、regime、model comparison 三组 sidecar，top model 为 mean_return_smoke。
- regime 差异：产物按 VIX、趋势、回撤、波动率、成交量冲击和牛熊/震荡切片，供后续模型稳定性分析复用。
- 是否晋升：不晋升；这是框架烟测卡，不是生产策略。
- 风险：样本和模型都只是基础验证，不能作为交易信号。
- 下一步：接入真实 QuantBench model registry、objective promotion score 和 production paper/sim notebook。
